# Raw data Check

## TWSE

In [3]:
import requests
import datetime
import time
import random
import pandas as pd
from tqdm import tqdm
import os

def get_web_content(url):
    resp = requests.get(url)
    if resp.status_code != 200:
        return None
    else:
        try:
            return resp.json()
        except:
            return None

### MI_INDEX_每日收盤行情

In [2]:
path = "Y:\\TWSE\\MI_INDEX_每日收盤行情"
data9_path = os.path.join(path, 'data9')
check_files = [f for f in sorted(os.listdir(data9_path)) if f.split('.')[0]>="20110801"]

In [3]:
for f in tqdm(check_files[::100]):
    date_ = f.split('.')[0]
    url = f"https://www.twse.com.tw/exchangeReport/MI_INDEX?response=json&date={date_}&type=ALL"
    request_data = get_web_content(url)

    if request_data['stat'] != '很抱歉，沒有符合條件的資料!':
        for dataid in range(9,10):
            writing_data = pd.DataFrame(request_data['data'+str(dataid)])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields'+str(dataid)]
                writing_data.columns = writing_data_columns

            tmp = pd.read_csv(os.path.join(os.path.join(path, 'data'+str(dataid)),f))
            
            if tmp.shape[0]!=writing_data.shape[0]:
                print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|██████████████████████████████████████████████████████████████████████████████████| 32/32 [02:17<00:00,  4.30s/it]


### Notice

In [4]:
start_date = '20090101'
url = f"https://www.twse.com.tw/announcement/notice?response=json&startDate={start_date}&stockNo=&sortKind=STKNO&querytype=1&selectType="
request_data = get_web_content(url)
if (request_data != None):
    if (request_data['stat'] == 'OK'):
        writing_data = pd.DataFrame(request_data['data'])
        if writing_data.shape[0] > 0:
            writing_data_columns = request_data['fields']
            writing_data.columns = writing_data_columns

In [5]:
datetime_list = writing_data['日期'].apply(lambda x: str(int(x.split('.')[0])+1911)+x.split('.')[1]+x.split('.')[2])

In [6]:
path = "Y:\\TWSE\\notice"
for f in tqdm(sorted(os.listdir(path))):
    date_ = f.split('.')[0]
    tmp = pd.read_csv(os.path.join(path,f))
    if tmp.shape[0]!=writing_data[datetime_list==date_].shape[0]:
        print(date_)
        # print(tmp,writing_data[datetime_list==date_])
        print()

100%|██████████████████████████████████████████████████████████████████████████████| 3528/3528 [00:41<00:00, 85.99it/s]


### TWT49U_除權除息計算結果表

In [7]:
def get_web_content(url):
    
    error_class = ""
    try:
        resp = requests.get(url)
    except Exception as e:
        error_class = e.__class__.__name__
        detail = e.args[0]
        print(error_class)
        print(detail)


    if error_class == "ConnectionError":
        
        print("Run Bat")
        os.startfile("VPN1.bat")
        time.sleep(15)
        
        return None
        
    
    if resp.status_code != 200:
        return None
    else:
        try:
            return resp.json()
        except:
            return None

In [8]:
start_date = '20100101'
url = f"https://www.twse.com.tw/rwd/zh/exRight/TWT49U?response=json&startDate={start_date}"

request_data = get_web_content(url)

if request_data is not None:

    if request_data['stat'] != '很抱歉，沒有符合條件的資料!':

        writing_data = pd.DataFrame()

        writing_data = pd.DataFrame(request_data['data'])
        

        
        
        if (writing_data.shape[0] > 0)&(writing_data.shape[1] > 3):
        
            writing_data_columns = request_data['fields']
            writing_data.columns = writing_data_columns
            writing_data['DateForCheck'] = writing_data['資料日期']
            writing_data['DateForCheck'] = writing_data['DateForCheck'].str.replace("年","/")
            writing_data['DateForCheck'] = writing_data['DateForCheck'].str.replace("月","/")
            writing_data['DateForCheck'] = writing_data['DateForCheck'].str.replace("日","")
            # writing_data['DateForCheck'] = writing_data['DateForCheck'].str.replace(" ","")
            
            
            if (writing_data.shape[0] > 0)&(writing_data.shape[1] > 3):
                
                try:
                    writing_data_columns = [w.replace('</br>', '') for w in writing_data_columns]
                except:
                    pass

In [9]:
path = "Y:\\TWSE\\TWT49U_除權除息計算結果表"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [10]:
dfs = [pd.read_csv(os.path.join(path,f)) for f in tqdm(files) if f.split('.')[0]>='20100101']

100%|█████████████████████████████████████████████████████████████████████████████| 1909/1909 [00:12<00:00, 154.08it/s]


In [11]:
tmp = pd.concat(dfs, axis=0)
tmp["股票代號"] = tmp["股票代號"].astype(str)

In [12]:
writing_data2 = writing_data.copy()

In [13]:
tmp = tmp.set_index(["資料日期", "股票代號"]).sort_index()
writing_data2 = writing_data2.set_index(["資料日期", "股票代號"]).sort_index()

In [14]:
for i in writing_data2[~writing_data2.index.isin(tmp.index)].index:
    if i[1][0]=='0':
        continue
    print(i)
    date_ = i[0].replace('年','').replace('月','').replace('日','')
    file = str(int(str(date_)[:-4])+1911)+str(date_)[-4:]+'.csv'
    print(file)
    # writing_data[writing_data['資料日期']==i[0]].reset_index(drop=True).to_csv(os.path.join(path, file))

### Punish

In [15]:
start_date = '20100101'
url = f"https://www.twse.com.tw/announcement/punish?response=json&startDate={start_date}&stockNo=&sortKind=STKNO&querytype=1&selectType="
request_data = get_web_content(url)
if (request_data != None):
    if (request_data['stat'] == 'OK'):
        writing_data = pd.DataFrame(request_data['data'])
        if writing_data.shape[0] > 0:
            writing_data_columns = request_data['fields']
            writing_data.columns = writing_data_columns

In [16]:
path = "Y:\\TWSE\\punish"

In [17]:
dfs = [pd.read_csv(os.path.join(path,f)) for f in tqdm(os.listdir(path))]

100%|██████████████████████████████████████████████████████████████████████████████| 4737/4737 [00:56<00:00, 84.50it/s]


In [18]:
nas_data = pd.concat(dfs, axis=0)
nas_data['證券代號'] = nas_data['證券代號'].astype(str)
nas_data = nas_data.drop_duplicates(subset=['公布日期','證券代號']).set_index(['公布日期','證券代號']).sort_index()

In [19]:
writing_data = writing_data.drop_duplicates(subset=['公布日期','證券代號']).sort_values(['公布日期','證券代號']).set_index(['公布日期','證券代號']).sort_index()

In [20]:
writing_data = writing_data.loc['106/05/08':]

In [21]:
len([i for i in nas_data.index if i in writing_data.index])

1133

In [22]:
len([i for i in nas_data.index if i not in writing_data.index]),len([i for i in writing_data.index if i not in nas_data.index])

(871, 192)

In [23]:
[i for i in nas_data.index if i not in writing_data.index][:5],[i for i in writing_data.index if i not in nas_data.index][:5]

([('100/01/04', '9157'),
  ('100/01/07', '9103'),
  ('100/01/11', '9157'),
  ('100/01/13', '6209'),
  ('100/01/14', '9103')],
 [('106/10/27', '064766'),
  ('106/11/13', '064656'),
  ('106/12/28', '062015'),
  ('107/01/09', '065750'),
  ('107/01/11', '063514')])

### BFI84U_得為融資融券有價證券停券預告表

In [26]:
path = "Y:\\TWSE\\BFI84U_得為融資融券有價證券停券預告表"
for f in tqdm(os.listdir(path)[:1]):
    date_ = f.split('.')[0]
    tmp = pd.read_csv(os.path.join(path,f))

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 62.50it/s]


In [27]:
url = f"https://www.twse.com.tw/exchangeReport/BFI84U?response=json&_={date_}"

In [28]:
request_data = get_web_content(url)
if (request_data != None):
    if (request_data['stat'] == 'OK'):
        writing_data = pd.DataFrame(request_data['data'])
        if writing_data.shape[0] > 0:
            writing_data_columns = request_data['fields']
            writing_data.columns = writing_data_columns

### BWIBBU_個股本益比殖利率股價淨值比

In [29]:
path = "Y:\\TWSE\\BWIBBU_個股本益比殖利率股價淨值比"
files = sorted(os.listdir(path))

In [30]:
# 寬鬆測
for f in tqdm(files[::100]):
    date_ = f.split('.')[0]
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/exchangeReport/BWIBBU_d?response=json&date={date_}&selectType=ALL"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        if tmp.shape[0]!=writing_data.shape[0]:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|██████████████████████████████████████████████████████████████████████████████████| 36/36 [02:10<00:00,  3.64s/it]


In [31]:
# 測20200101後 嚴謹測
for f in tqdm(files[::18]):
    date_ = f.split('.')[0]
    if date_<='20200101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/exchangeReport/BWIBBU_d?response=json&date={date_}&selectType=ALL"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        if tmp.shape[0]!=writing_data.shape[0]:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 6))

100%|████████████████████████████████████████████████████████████████████████████████| 195/195 [04:10<00:00,  1.28s/it]


### MI_5MINS_INDEX_每5秒指數統計

In [32]:
path = "Y:\\TWSE\\MI_5MINS_INDEX_每5秒指數統計"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [33]:
tw = pd.read_csv('D:\\DerivativesChina\\05data\\POW_index.csv')

In [34]:
# 測20200101後 嚴謹測
for f in tqdm(files):
    date_ = f.split('.')[0]
    # if date_<='20200101':
    #     continue
    tmp = pd.read_csv(os.path.join(path,f))
    if tw[tw['Date'].apply(lambda x: x.replace('-',''))==date_].shape[0]==0:
        continue
    if abs(float(tmp['發行量加權股價指數'].iloc[-1].replace(',',''))/tw[tw['Date'].apply(lambda x: x.replace('-',''))==date_]['Close'].iloc[0]-1)>0.0001:
        print(date_)

100%|██████████████████████████████████████████████████████████████████████████████| 1507/1507 [01:27<00:00, 17.28it/s]


### MI_MARGN_信用交易統計

In [35]:
path = "Y:\\TWSE\\MI_MARGN_信用交易統計"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [36]:
# 全範圍 寬鬆測
for f in tqdm(files[::100]):
    date_ = f.split('.')[0]
    # if date_<='20200101':
    #     continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/exchangeReport/MI_MARGN?response=json&date={date_}&selectType=ALL"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        cond1 = tmp.shape[0]!=writing_data.shape[0]
        cond2 = tmp['股票名稱'].iloc[0]!=writing_data['股票名稱'].iloc[0]
        cond3 = tmp['股票名稱'].iloc[-1]!=writing_data['股票名稱'].iloc[-1]
        cond4 = tmp['股票名稱'].iloc[10]!=writing_data['股票名稱'].iloc[10]
        if cond1 or cond2 or cond3 or cond4:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|██████████████████████████████████████████████████████████████████████████████████| 36/36 [02:30<00:00,  4.18s/it]


In [37]:
# 測20200101後 嚴謹測
for f in tqdm(files[::18]):
    date_ = f.split('.')[0]
    # if date_<='20200101':
    #     continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/exchangeReport/MI_MARGN?response=json&date={date_}&selectType=ALL"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        cond1 = tmp.shape[0]!=writing_data.shape[0]
        cond2 = tmp['股票名稱'].iloc[0]!=writing_data['股票名稱'].iloc[0]
        cond3 = tmp['股票名稱'].iloc[-1]!=writing_data['股票名稱'].iloc[-1]
        cond4 = tmp['股票名稱'].iloc[10]!=writing_data['股票名稱'].iloc[10]
        if cond1 or cond2 or cond3 or cond4:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|████████████████████████████████████████████████████████████████████████████████| 195/195 [12:54<00:00,  3.97s/it]


### MI_QFIIS_外資及陸資投資持股統計

In [38]:
path = "Y:\\TWSE\\MI_QFIIS_外資及陸資投資持股統計"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [39]:
# 全範圍 寬鬆測
for f in tqdm(files[::100]):
    date_ = f.split('.')[0]
    if date_<='20140101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/rwd/zh/fund/MI_QFIIS?response=json&date={date_}&selectType=ALLBUT0999"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        cond1 = tmp.shape[0]!=writing_data.shape[0]
        check_col = "外資及陸資尚可投資股數"
        cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
        cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
        cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
        if cond1 or cond2 or cond3 or cond4:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|██████████████████████████████████████████████████████████████████████████████████| 41/41 [01:50<00:00,  2.70s/it]


In [40]:
# 測20200101後 嚴謹測
for f in tqdm(files[::19]):
    date_ = f.split('.')[0]
    if date_<='20200101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/rwd/zh/fund/MI_QFIIS?response=json&date={date_}&selectType=ALLBUT0999"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        cond1 = tmp.shape[0]!=writing_data.shape[0]
        check_col = "外資及陸資尚可投資股數"
        cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
        cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
        cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
        if cond1 or cond2 or cond3 or cond4:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|████████████████████████████████████████████████████████████████████████████████| 214/214 [03:38<00:00,  1.02s/it]


### T86_三大法人買賣超日報

In [41]:
path = "Y:\\TWSE\\T86_三大法人買賣超日報"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [42]:
# 全範圍 寬鬆測
for f in tqdm(files[::100]):
    date_ = f.split('.')[0]
    if date_>='20140101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/rwd/zh/fund/T86?response=json&date={date_}&selectType=ALL"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        cond1 = tmp.shape[0]!=writing_data.shape[0]
        check_col = "外資買進股數"
        cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
        cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
        cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
        if cond1 or cond2 or cond3 or cond4:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [00:20<00:00,  1.47it/s]


In [43]:
# 全範圍 寬鬆測
for f in tqdm(files[::100]):
    date_ = f.split('.')[0]
    if date_<='20180101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/rwd/zh/fund/T86?response=json&date={date_}&selectType=ALL"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        cond1 = tmp.shape[0]!=writing_data.shape[0]
        check_col = "外陸資買進股數(不含外資自營商)"
        cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
        cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
        cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
        if cond1 or cond2 or cond3 or cond4:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [01:16<00:00,  2.54s/it]


In [44]:
# 全範圍 寬鬆測
for f in tqdm(files[::100]):
    date_ = f.split('.')[0]
    if date_<='20200101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/rwd/zh/fund/T86?response=json&date={date_}&selectType=ALL"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        cond1 = tmp.shape[0]!=writing_data.shape[0]
        check_col = "外陸資買進股數(不含外資自營商)"
        cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
        cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
        cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
        if cond1 or cond2 or cond3 or cond4:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [00:51<00:00,  1.73s/it]


### TWT93U_信用額度總量管制餘額表

In [45]:
path = "Y:\\TWSE\\TWT93U_信用額度總量管制餘額表"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [55]:
# 全範圍 寬鬆測
for f in tqdm(files[::100]):
    date_ = f.split('.')[0]
    if date_<='20150101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/rwd/zh/marginTrading/TWT93U?response=json&date={date_}"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        cond1 = tmp.shape[0]!=writing_data.shape[0]
        check_col1 = "限額"
        check_col = "次一營業日限額"
        cond2 = tmp[check_col1].iloc[0]!=writing_data[check_col].iloc[0]
        cond3 = tmp[check_col1].iloc[-1]!=writing_data[check_col].iloc[-1]
        cond4 = tmp[check_col1].iloc[10]!=writing_data[check_col].iloc[10]
        if cond1 or cond2 or cond3 or cond4:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|██████████████████████████████████████████████████████████████████████████████████| 41/41 [01:30<00:00,  2.20s/it]


In [57]:
# 測20200101後 嚴謹測
for f in tqdm(files[::18]):
    date_ = f.split('.')[0]
    if date_<='20200101' or date_>='20240320':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/rwd/zh/marginTrading/TWT93U?response=json&date={date_}"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        cond1 = tmp.shape[0]!=writing_data.shape[0]
        check_col1 = "限額"
        check_col = "次一營業日限額"
        cond2 = tmp[check_col1].iloc[0]!=writing_data[check_col].iloc[0]
        cond3 = tmp[check_col1].iloc[-1]!=writing_data[check_col].iloc[-1]
        cond4 = tmp[check_col1].iloc[10]!=writing_data[check_col].iloc[10]
        if cond1 or cond2 or cond3 or cond4:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|████████████████████████████████████████████████████████████████████████████████| 224/224 [03:51<00:00,  1.03s/it]


In [59]:
# 測20200101後 嚴謹測
for f in tqdm(files[::18]):
    date_ = f.split('.')[0]
    if date_<='20240401':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/rwd/zh/marginTrading/TWT93U?response=json&date={date_}"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        cond1 = tmp.shape[0]!=writing_data.shape[0]
        check_col = "次一營業日限額"
        cond2 = tmp[check_col1].iloc[0]!=writing_data[check_col].iloc[0]
        cond3 = tmp[check_col1].iloc[-1]!=writing_data[check_col].iloc[-1]
        cond4 = tmp[check_col1].iloc[10]!=writing_data[check_col].iloc[10]
        if cond1 or cond2 or cond3 or cond4:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|████████████████████████████████████████████████████████████████████████████████████████| 224/224 [00:00<?, ?it/s]


### TWTB4U_當日沖銷交易統計資訊

In [60]:
path = "Y:\\TWSE\\TWTB4U_當日沖銷交易統計資訊"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [61]:
# 測20200101後 嚴謹測
for f in tqdm(files[::-1][::20]):
    date_ = f.split('.')[0]
    if date_<='20200101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/exchangeReport/TWTB4U?response=json&date={date_}&selectType=All"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                try:
                    writing_data_columns = [w.replace('</br>', '') for w in writing_data_columns]
                except:
                    pass
                writing_data.columns = writing_data_columns
        cond1 = tmp.shape[0]!=writing_data.shape[0]
        check_col = "限額"
        try:
            cond2 = tmp[tmp['暫停現股賣出後現款買進當沖註記']=='Y'].shape[0]!=writing_data[writing_data['暫停現股賣出後現款買進當沖註記']=='Y'].shape[0]
            if cond1 or cond2:
                print(tmp,writing_data)
        except:
            pass
    time.sleep(random.randint(2, 5))

100%|██████████████████████████████████████████████████████████████████████████████████| 74/74 [03:38<00:00,  2.95s/it]


### TWT84U_股價升降幅度

In [62]:
path = "Y:\\TWSE\\TWT84U_股價升降幅度"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [63]:
# 全範圍 寬鬆測
for f in tqdm(files[::100]):
    date_ = f.split('.')[0]
    if date_<='20130101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/exchangeReport/TWT84U?response=json&date={date_}&selectType=ALL"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        cond1 = tmp.shape[0]!=writing_data.shape[0]
        check_col = "漲停價"
        cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
        cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
        cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
        if cond1 or cond2:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|██████████████████████████████████████████████████████████████████████████████████| 36/36 [02:10<00:00,  3.63s/it]


In [64]:
# 測20200101後 嚴謹測
for f in tqdm(files[::28]):
    date_ = f.split('.')[0]
    if date_<='20200101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    url = f"https://www.twse.com.tw/exchangeReport/TWT84U?response=json&date={date_}&selectType=ALL"
    
    request_data = get_web_content(url)
    if (request_data != None):
        if (request_data['stat'] == 'OK'):
            writing_data = pd.DataFrame(request_data['data'])
            if writing_data.shape[0] > 0:
                writing_data_columns = request_data['fields']
                writing_data.columns = writing_data_columns
        cond1 = tmp.shape[0]!=writing_data.shape[0]
        check_col = "漲停價"
        cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
        cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
        cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
        if cond1 or cond2:
            print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|████████████████████████████████████████████████████████████████████████████████| 126/126 [02:50<00:00,  1.35s/it]


## TPEX

### daily_close_quotes

In [4]:
def to_float(string):
    if isinstance(string, float):
        return string
    elif string == '-' or string == '--' or string == '---' or string == '----' or string == 'nan' or string == ' ---':
        return np.nan
    else:
        return float(string.replace(",",""))

In [5]:
def ce_to_roc(dt, seperator='/'):
    return str(int(dt.split(seperator)[0]) - 1911) + dt[4:]

In [67]:
path = "Y:\\TPEX\\aftertrading\\daily_close_quotes"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [68]:
cols = ['證券代號', '證券名稱', '收盤', '漲跌', '開盤', '最高', '最低', '均價', '成交股數',
        '成交金額(元)', '成交筆數', '最後買價', '最後買量(千股)', '最後賣價', '最後賣量(千股)',
        '發行股數', '次日參考價', '次日漲停價', '次日跌停價']

In [69]:
# 全範圍 寬鬆測
for f in tqdm(files[::100]):
    date_ = f.split('.')[0]
    if date_<='20180101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    get_dt_roc = ce_to_roc(pd.to_datetime(date_).strftime("%Y/%m/%d"))
    url = f"https://www.tpex.org.tw/web/stock/aftertrading/daily_close_quotes/stk_quote_result.php?l=zh-tw&d={get_dt_roc}"
    request_data = get_web_content(url)
    
    writing_data = pd.DataFrame(request_data['aaData'])
    
    writing_data_mm = pd.DataFrame()
    writing_data_mm = pd.DataFrame(request_data['mmData'])
    
    if writing_data_mm.shape[0] > 0:
        writing_data = pd.concat([writing_data,writing_data_mm])
    if writing_data.shape[0] > 0:
        writing_data.columns = cols
        
    cond1 = tmp.shape[0]!=writing_data.shape[0]
    check_col = "收盤"
    cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
    cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
    cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
    if cond1 or cond2:
        print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|██████████████████████████████████████████████████████████████████████████████████| 36/36 [01:05<00:00,  1.81s/it]


In [70]:
# 測20200101後 嚴謹測
for f in tqdm(files[::18]):
    date_ = f.split('.')[0]
    if date_<='20200101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    get_dt_roc = ce_to_roc(pd.to_datetime(date_).strftime("%Y/%m/%d"))
    url = f"https://www.tpex.org.tw/web/stock/aftertrading/daily_close_quotes/stk_quote_result.php?l=zh-tw&d={get_dt_roc}"
    request_data = get_web_content(url)
    
    writing_data = pd.DataFrame(request_data['aaData'])
    
    writing_data_mm = pd.DataFrame()
    writing_data_mm = pd.DataFrame(request_data['mmData'])
    
    if writing_data_mm.shape[0] > 0:
        writing_data = pd.concat([writing_data,writing_data_mm])
    if writing_data.shape[0] > 0:
        writing_data.columns = cols
        
    cond1 = tmp.shape[0]!=writing_data.shape[0]
    check_col = "收盤"
    cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
    cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
    cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
    if cond1 or cond2:
        print(tmp,writing_data)
    time.sleep(random.randint(3, 6))

100%|████████████████████████████████████████████████████████████████████████████████| 195/195 [04:48<00:00,  1.48s/it]


### peratio_analysis_個股本益比殖利率股價淨值比

In [71]:
path = "Y:\\TPEX\\aftertrading\\peratio_analysis_個股本益比殖利率股價淨值比"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [72]:
cols = ['證券代號', '名稱', '本益比', '每股股利', '股利年度', '殖利率(%)', '股價淨值比']

In [73]:
# 測20200101後 嚴謹測
for f in tqdm(files[::20]):
    date_ = f.split('.')[0]
    if date_<='20200101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    get_dt_roc = ce_to_roc(pd.to_datetime(date_).strftime("%Y/%m/%d"))
    url = "https://www.tpex.org.tw/web/stock/aftertrading/peratio_analysis/pera_result.php?l=zh-tw&d={}&c=".format(get_dt_roc)
    request_data = get_web_content(url)
    
    writing_data = pd.DataFrame()
    writing_data = pd.DataFrame(request_data['aaData'])
    if writing_data.shape[0] > 0:
               
        writing_data.columns = cols
        
    cond1 = tmp.shape[0]!=writing_data.shape[0]
    check_col = '本益比'
    tmp[check_col] = tmp[check_col].astype(str)
    cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
    cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
    cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
    if cond1 or cond2 or cond3 or cond4:
        print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

  0%|                                                                                          | 0/176 [00:00<?, ?it/s]

     Unnamed: 0  證券代號     名稱    本益比       每股股利  股利年度  殖利率(%)  股價淨值比
0             0  1240   茂生農經  12.97   2.300000   108    3.33   1.83
1             1  1258  其祥-KY    nan   0.000000   108    0.00   0.99
2             2  1259     安心  15.08   3.200000   108    3.95   1.35
3             3  1264     德麥  16.36  11.003919   108    4.94   3.34
4             4  1268   漢來美食  19.34   7.000000   108    4.98   3.61
..          ...   ...    ...    ...        ...   ...     ...    ...
767         767  9949     琉園    nan   0.000000   108    0.00   1.16
768         768  9950    萬國通    nan   0.000000   108    0.00   1.66
769         769  9951     皇田  11.28   3.500000   108    4.75   2.43
770         770  9960    邁達康  32.71   0.800000   108    1.43   2.32
771         771  9962     有益   98.3   0.200000   108    9.16   0.94

[772 rows x 8 columns]      證券代號     名稱    本益比         每股股利 股利年度 殖利率(%) 股價淨值比
0    1240   茂生農經  12.97   2.30000000  108   3.33  1.83
1    1258  其祥-KY    N/A   0.00000000  108   0.00  

 71%|████████████████████████████████████████████████████████▊                       | 125/176 [00:02<00:00, 54.66it/s]

     Unnamed: 0  證券代號     名稱    本益比       每股股利  股利年度  殖利率(%)  股價淨值比
0             0  1240   茂生農經  12.64   2.300000   108    3.42   1.78
1             1  1258  其祥-KY    nan   0.000000   108    0.00   0.99
2             2  1259     安心  14.74   3.200000   108    4.04   1.32
3             3  1264     德麥  15.85  11.003919   108    5.10   3.23
4             4  1268   漢來美食  17.89   7.000000   108    5.38   3.34
..          ...   ...    ...    ...        ...   ...     ...    ...
766         766  9949     琉園    nan   0.000000   108    0.00   1.15
767         767  9950    萬國通    nan   0.000000   108    0.00   1.54
768         768  9951     皇田  10.01   3.500000   108    5.35   2.16
769         769  9960    邁達康  31.25   0.800000   108    1.50   2.22
770         770  9962     有益   96.1   0.200000   108    9.37   0.92

[771 rows x 8 columns]      證券代號     名稱    本益比         每股股利 股利年度 殖利率(%) 股價淨值比
0    1240   茂生農經  12.64   2.30000000  108   3.42  1.78
1    1258  其祥-KY    N/A   0.00000000  108   0.00  

 74%|███████████████████████████████████████████████████████████                     | 130/176 [00:21<00:10,  4.43it/s]

...

 99%|███████████████████████████████████████████████████████████████████████████████ | 174/176 [03:01<00:06,  3.19s/it]

     Unnamed: 0  證券代號    名稱    本益比   每股股利  股利年度  殖利率(%)  股價淨值比
0             0  1240  茂生農經  17.73   1.20   111    2.42   1.53
1             1  1259    安心  13.85   1.80   112    3.43   1.01
2             2  1264    德麥  16.57  12.00   111    4.00   3.68
3             3  1268  漢來美食  17.34   3.00   111    1.95   3.15
4             4  1336    台翰  18.68   0.41   111    1.81   0.98
..          ...   ...   ...    ...    ...   ...     ...    ...
806         806  9949    琉園    nan   0.00   111    0.00   2.13
807         807  9950   萬國通   3.31   0.00   111    0.00   2.51
808         808  9951    皇田  11.76   3.30   111    4.57   1.53
809         809  9960   邁達康   9.52   2.20   111    8.26   1.47
810         810  9962    有益  10.56   1.80   111   10.03   1.42

[811 rows x 8 columns]      證券代號    名稱    本益比         每股股利 股利年度 殖利率(%) 股價淨值比
0    1240  茂生農經  17.73   2.70000000  112   2.42  1.53
1    1259    安心  13.85   1.80000000  112   3.43  1.01
2    1264    德麥  16.57  14.00000000  112   4.00  3.68
3   

 99%|███████████████████████████████████████████████████████████████████████████████▌| 175/176 [03:04<00:03,  3.22s/it]

     Unnamed: 0  證券代號    名稱    本益比   每股股利  股利年度  殖利率(%)  股價淨值比
0             0  1240  茂生農經  18.32   2.70   112    5.26   1.58
1             1  1259    安心  14.51   1.80   112    2.40   1.03
2             2  1264    德麥  16.72  14.00   112    4.59   3.59
3             3  1268  漢來美食  20.24   3.00   111    1.95   3.02
4             4  1336    台翰   14.6   0.41   112    2.02   0.90
..          ...   ...   ...    ...    ...   ...     ...    ...
811         811  9949    琉園    nan   0.00   112    0.00   2.23
812         812  9950   萬國通   2.79   0.00   112    0.00   2.13
813         813  9951    皇田  12.51   3.40   112    4.71   1.50
814         814  9960   邁達康  10.67   1.70   112    6.32   1.43
815         815  9962    有益  13.93   1.10   112    6.27   1.37

[816 rows x 8 columns]      證券代號    名稱    本益比         每股股利 股利年度 殖利率(%) 股價淨值比
0    1240  茂生農經  18.32   2.70000000  112   5.26  1.58
1    1259    安心  14.51   1.80000000  112   2.40  1.03
2    1264    德麥  16.72  14.00000000  112   4.59  3.59
3   

100%|████████████████████████████████████████████████████████████████████████████████| 176/176 [03:08<00:00,  1.07s/it]


### cmode_變更交易、分盤交易、管理股票與停止交易資訊

In [74]:
path = "Y:\\TPEX\\aftertrading\\cmode_變更交易、分盤交易、管理股票與停止交易資訊"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [75]:
cols = ["證券代號", "證券名稱", "變更交易", "分盤交易", "署管理股票", "分盤或管理股票撮合循環時間(分鐘)", "停止交易",
        "財務重點專區", "unknown1", "unknown2"]

In [76]:
# 測20200101後 嚴謹測
for f in tqdm(files[::18][-30:]):
    date_ = f.split('.')[0]
    if date_<='20180101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    get_dt_roc = ce_to_roc(pd.to_datetime(date_).strftime("%Y/%m/%d"))
    url = "https://www.tpex.org.tw/web/stock/aftertrading/cmode/chtm_result.php?l=zh-tw&o=json&d={}".format(get_dt_roc)

    request_data = get_web_content(url)
    
    writing_data = pd.DataFrame()
    writing_data = pd.DataFrame(request_data['aaData'])
    if writing_data.shape[0] > 0:
               
        writing_data.columns = cols
        
    cond1 = tmp.shape[0]!=writing_data.shape[0]
    check_col = '變更交易'
    tmp[check_col] = tmp[check_col].astype(str)
    cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
    cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
    cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
    if cond1 or cond2 or cond3 or cond4:
        print(tmp,writing_data)
    time.sleep(random.randint(4, 5))

100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [02:34<00:00,  5.16s/it]


### margin_balance_融資融券餘額

In [77]:
path = "Y:\\TPEX\\margin_trading\\margin_balance_融資融券餘額"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [78]:
cols = ["證券代號", "證券名稱", "前資餘額(張)", "資買", "資賣", "現償", "資餘額", "資屬證金", "資使用率(%)", "資限額",
        "前券餘額(張)", "券賣", "券買", "券償", "券餘額", "券屬證金", "券使用率(%)", "券限額", "資券相抵(張)", "備註"]

In [79]:
# 測20200101後 嚴謹測
for f in tqdm(files[::18]):
    date_ = f.split('.')[0]
    if date_<='20200101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    get_dt_roc = ce_to_roc(pd.to_datetime(date_).strftime("%Y/%m/%d"))
    url = "https://www.tpex.org.tw/web/stock/margin_trading/margin_balance/margin_bal_result.php?l=zh-tw&o=json&d={}".format(get_dt_roc)

    request_data = get_web_content(url)
    
    writing_data = pd.DataFrame()
    writing_data = pd.DataFrame(request_data['aaData'])
    if writing_data.shape[0] > 0:
               
        writing_data.columns = cols
        
    cond1 = tmp.shape[0]!=writing_data.shape[0]
    check_col = '前資餘額(張)'
    tmp[check_col] = tmp[check_col].astype(str)
    cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
    cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
    cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
    if cond1 or cond2 or cond3 or cond4:
        print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|████████████████████████████████████████████████████████████████████████████████| 153/153 [03:42<00:00,  1.46s/it]


### margin_sbl_信用額度總量管制餘額

In [6]:
path = "Y:\\TPEX\\margin_trading\\margin_sbl_信用額度總量管制餘額"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [7]:
cols = ["證券代號", "證券名稱", "融券_前日餘額", "融券_賣出", "融券_買進", "融券_現券", "融券_當日餘額", "融券_限額", 
         "借券賣出_前日餘額", "借券賣出_當日賣出", "借券賣出_當日還券", "借券賣出_當日調整數額", 
         "借券賣出_當日餘額", "借券賣出_次一營業日可借券賣出限額", "備註"]

In [8]:
# 測20200101後 嚴謹測
for f in tqdm(files[::18]):
    date_ = f.split('.')[0]
    if date_<='20180101':
        continue
    tmp = pd.read_csv(os.path.join(path,f))
    get_dt_roc = ce_to_roc(pd.to_datetime(date_).strftime("%Y/%m/%d"))
    url = "https://www.tpex.org.tw/web/stock/margin_trading/margin_sbl/margin_sbl_result.php?l=zh-tw&d={}".format(get_dt_roc)

    request_data = get_web_content(url)
    
    writing_data = pd.DataFrame()
    writing_data = pd.DataFrame(request_data['aaData'])
    if writing_data.shape[0] > 0:
               
        writing_data.columns = cols
        
    cond1 = tmp.shape[0]!=writing_data.shape[0]
    check_col = '融券_前日餘額'
    tmp[check_col] = tmp[check_col].astype(str)
    cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
    cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
    cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
    if cond1 or cond2 or cond3 or cond4:
        print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|████████████████████████████████████████████████████████████████████████████████| 152/152 [05:11<00:00,  2.05s/it]


### daily_trade_三大法人買賣明細資訊

In [9]:
path = "Y:\\TPEX\\3insti\\daily_trade_三大法人買賣明細資訊"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [10]:
insti = ["外資及陸資(不含外資自營商)", "外資自營商", "外資及陸資", "投信", "自營商(自行買賣)", "自營商(避險)", "自營商"]
sub = ["買進股數", "賣出股數", "買賣超股數"]
title = [i+j for i in insti for j in sub]
title = ["證券代號", "證券名稱"] + title + ["三大法人買賣超股數合計", "unknown"]

insti2 = ["外資及陸資", "投信", "自營商(自行買賣)", "自營商(避險)"]
sub2 = ["買進股數", "賣出股數", "買賣超股數"]
title2 = [i+j for i in insti2 for j in sub2]
title2 = ["證券代號", "證券名稱"] + title2[0:6] + ["自營商買賣超股數"] + title2[6::] + ["三大法人買賣超股數合計", "unknown"]

In [11]:
# 測20200101後 嚴謹測
for f in tqdm(files[::18]):
    date_ = f.split('.')[0]
    if date_<='20200101':
        continue
    if date_ < "20180114":
        cols = title2
    else:
        cols = title
    tmp = pd.read_csv(os.path.join(path,f))
    get_dt_roc = ce_to_roc(pd.to_datetime(date_).strftime("%Y/%m/%d"))
    url = "https://www.tpex.org.tw/web/stock/3insti/daily_trade/3itrade_hedge_result.php?l=zh-tw&se=AL&t=D&d={}".format(get_dt_roc)


    request_data = get_web_content(url)
    
    writing_data = pd.DataFrame()
    writing_data = pd.DataFrame(request_data['aaData'])
    if writing_data.shape[0] > 0:
               
        writing_data.columns = cols
        
    cond1 = tmp.shape[0]!=writing_data.shape[0]
    check_col = '三大法人買賣超股數合計'
    tmp[check_col] = tmp[check_col].astype(str)
    cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
    cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
    cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
    if cond1 or cond2 or cond3 or cond4:
        print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|████████████████████████████████████████████████████████████████████████████████| 127/127 [03:56<00:00,  1.86s/it]


### t13sa150_otc_外資及陸資投資持股統計

In [12]:
path = "Y:\TPEX\mops\\t13sa150_otc_外資及陸資投資持股統計"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [13]:
col_name = ['證券代號','證券名稱','發行股數','外資及陸資尚可投資股數','全體外資及陸資持有股數','外資及陸資尚可投資比率','全體外資及陸資持股比率',
       '外資及陸資共用法令投資上限比率','陸資法令投資上限比率','與前日異動原因(註)','最近一次上櫃公司申報外資持股異動日期']

In [14]:
def get_web_content(url):
    
    error_class = ""
    try:
        resp = requests.get(url)
        
        resp.encoding = 'big5'
    except Exception as e:
        error_class = e.__class__.__name__
        detail = e.args[0]
        print(error_class)
        print(detail)


    if error_class == "ConnectionError":
        
        print("Run Bat")
        os.startfile("VPN1.bat")
        time.sleep(15)
        
        return None
        
    
    if resp.status_code != 200:
        return None
    else:
        try:
            return resp
        except:
            return None

In [15]:
from bs4 import BeautifulSoup
# 測20200101後 嚴謹測
for f in tqdm(files[::20]):
    date_ = f.split('.')[0]
    if date_<='20200101':
        continue
    if date_ < "20180114":
        cols = title2
    else:
        cols = title
    tmp = pd.read_csv(os.path.join(path,f))
    get_dt_YYYY = pd.to_datetime(date_).strftime("%Y")
    get_dt_MM = pd.to_datetime(date_).strftime("%m")
    get_dt_DD = pd.to_datetime(date_).strftime("%d")
    url = 'https://mops.twse.com.tw/server-java/t13sa150_otc?&step=2&years={}&months={}&days={}'.format(get_dt_YYYY,get_dt_MM,get_dt_DD)
    request_data = get_web_content(url)            

    writing_data = pd.DataFrame()
    # writing_data = pd.DataFrame(request_data['aaData'])
    
    soup = BeautifulSoup(request_data.text,'html.parser')
    tables = soup.find('table')
    
    data = []
    rows = tables.find_all('tr')
    
    for row in rows:
        cols = row.find_all('td')
        # cols = [ele.text.strip() for ele in cols]
        cols = [ele.text for ele in cols]
        data.append([ele for ele in cols if ele]) # Get rid of empty values
    
    writing_data = pd.DataFrame(data)   
    
    writing_data.columns = col_name
    
    writing_data = writing_data.dropna(how='all',axis=0)   
    symbol_list = list(set(tmp['證券代號'].tolist()+writing_data['證券代號'].tolist()))
    
    check_col = '外資及陸資尚可投資股數'
    cond1 = len(symbol_list)<700
    cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
    cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
    cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
    if cond1 or cond2 or cond3 or cond4:
        print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|████████████████████████████████████████████████████████████████████████████████| 139/139 [04:03<00:00,  1.76s/it]


### intraday_stat_現股當沖交易統計資訊

In [16]:
path = "Y:\\TPEX\\trading\\intraday_stat_現股當沖交易統計資訊"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [17]:
title1 = ["證券代號", "證券名稱", "當日沖銷交易成交股數", "當日沖銷交易買進成交金額", "當日沖銷交易賣出成交金額"]
title2 = ["證券代號", "證券名稱", "暫停現股賣出後現款買進當沖註記", "當日沖銷交易成交股數", "當日沖銷交易買進成交金額", 
          "當日沖銷交易賣出成交金額"]

In [18]:
def get_web_content(url):
    
    error_class = ""
    try:
        resp = requests.get(url)
    except Exception as e:
        error_class = e.__class__.__name__
        detail = e.args[0]
        print(error_class)
        print(detail)


    if error_class == "ConnectionError":
        
        print("Run Bat")
        os.startfile("VPN1.bat")
        time.sleep(15)
        
        return None
        
    
    if resp.status_code != 200:
        return None
    else:
        try:
            return resp.json()
        except:
            return None

In [19]:
# 測20200101後 嚴謹測
for f in tqdm(files[::18]):
    date_ = f.split('.')[0]
    if date_ < "20140629":
        cols = title1
    else:
        cols = title2    

    tmp = pd.read_csv(os.path.join(path,f))
    get_dt_roc = ce_to_roc(pd.to_datetime(date_).strftime("%Y/%m/%d"))
    url = "https://www.tpex.org.tw/web/stock/trading/intraday_stat/intraday_trading_stat_result.php?l=zh-tw&d={}".format(get_dt_roc)

    request_data = get_web_content(url)
    
    writing_data = pd.DataFrame()
    writing_data = pd.DataFrame(request_data['aaData'])
    if writing_data.shape[0] > 0:
               
        writing_data.columns = cols
        
    cond1 = tmp.shape[0]!=writing_data.shape[0]
    check_col = '當日沖銷交易成交股數'
    cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
    cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
    cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
    if cond1 or cond2 or cond3 or cond4:
        print(tmp,writing_data)
    time.sleep(random.randint(2, 5))

100%|████████████████████████████████████████████████████████████████████████████████| 139/139 [08:51<00:00,  3.82s/it]


### disposal_information_處置有價證券

In [20]:
path = "Y:\\TPEX\\bulletin\\disposal_information_處置有價證券"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files if f.split('.')[0]<'20200101']

In [21]:
dfs = [pd.read_csv(os.path.join(path,f)) for f in tqdm(files)]

100%|██████████████████████████████████████████████████████████████████████████████| 1476/1476 [00:54<00:00, 27.23it/s]


In [22]:
tmp = pd.concat(dfs, axis=0)
tmp['證券代號'] = tmp['證券代號'].astype(str)

In [24]:
# 測20200101後 嚴謹測
url = "https://www.tpex.org.tw/web/bulletin/disposal_information/disposal_information_print.php?l=zh-tw&sd=105/01/06&ed=112/09/07&code=&choice_type=all_category&stk_cotegory=-1&disposal_measure=-1&group_type=group_dat".format(get_dt_roc,get_dt_roc)
request_data = get_web_content(url)


punish = pd.read_html(request_data.text)[0]
punish = punish[punish["證券代號"].isna()==False].copy()
punish = punish.iloc[:-1]
punish[["start", "end"]] = punish["處置起訖時間"].str.split("~", 1, expand=True)
punish["start"] = [str(int(i.split("/")[0]) + 1911) + i[3::] for i in punish["start"]]
punish["end"] = [str(int(i.split("/")[0]) + 1911) + i[3::] for i in punish["end"]]
punish["start"] = pd.to_datetime(punish["start"])
punish["start"] = punish["start"].dt.strftime("%Y%m%d")
punish["end"] = pd.to_datetime(punish["end"])
punish["end"] = punish["end"].dt.strftime("%Y%m%d")

punish["trade_date"] = [str(int(i.split("/")[0]) + 1911) + i[3::] for i in punish["公布日期"]]
punish["trade_date"] = pd.to_datetime(punish["trade_date"])
punish["trade_date"] = punish["trade_date"].dt.strftime("%Y%m%d")


try:
    writing_data = pd.DataFrame()
    writing_data = pd.DataFrame(punish)
except:
    writing_data = pd.DataFrame()

if writing_data.shape[0] > 0:
           
    # writing_data.columns = cols
    
    writing_data_date = pd.DataFrame(writing_data["trade_date"].unique())
    writing_data_date = writing_data_date.sort_values(0,ascending=False)
    
# cond1 = tmp.shape[0]!=writing_data.shape[0]
# check_col = '當日沖銷交易成交股數'
# cond2 = tmp[check_col].iloc[0]!=writing_data[check_col].iloc[0]
# cond3 = tmp[check_col].iloc[-1]!=writing_data[check_col].iloc[-1]
# cond4 = tmp[check_col].iloc[10]!=writing_data[check_col].iloc[10]
# if cond1 or cond2 or cond3 or cond4:
#     print(tmp,writing_data)
# time.sleep(random.randint(2, 5))

AttributeError: 'NoneType' object has no attribute 'text'

In [438]:
tmp = tmp.set_index(['公布日期', '證券代號']).sort_index()

In [449]:
punish2 = punish.set_index(['公布日期', '證券代號']).sort_index()

In [450]:
len([i for i in tmp.index if i not in punish2.index]), len([i for i in punish2.index if i not in tmp.index])

(0, 14)

In [469]:
# 補齊數據
for l in [i for i in punish2.index if i not in tmp.index]:
    print(l)
    d = l[0]
    f = str(int(d.split('/')[0])+1911)+d.split('/')[1]+d.split('/')[2]+'.csv'
    print(f)
    punish[punish['公布日期']==d].to_csv(os.path.join(path, f))

('111/12/15', '3228')
20221215.csv
('112/04/17', '6465')
20230417.csv
('112/04/19', '26302')
20230419.csv
('112/04/19', '4728')
20230419.csv
('112/04/26', '4147')
20230426.csv
('112/05/05', '6752')
20230505.csv
('112/05/08', '3664')
20230508.csv
('112/06/14', '4533')
20230614.csv
('112/07/10', '30184')
20230710.csv
('112/07/10', '8093')
20230710.csv
('112/07/11', '6538')
20230711.csv
('112/07/19', '3483')
20230719.csv
('112/07/19', '34833')
20230719.csv
('112/08/02', '13165')
20230802.csv


### attention_information_注意有價證券

In [25]:
path = "Y:\\TPEX\\bulletin\\attention_information_注意有價證券"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files if f.split('.')[0]<'20200101']

In [26]:
dfs = [pd.read_csv(os.path.join(path,f)) for f in tqdm(files)]

100%|██████████████████████████████████████████████████████████████████████████████| 3504/3504 [00:36<00:00, 96.16it/s]


In [27]:
tmp = pd.concat(dfs, axis=0)
tmp['證券代號'] = tmp['證券代號'].astype(str)

In [28]:
cols = ['編號','證券代號','證券名稱','累計','注意交易資訊','公告日期','收盤價','本益比','trade_date']

In [30]:
# 測20200101後 嚴謹測
url = "https://www.tpex.org.tw/web/bulletin/attention_information/trading_attention_information_print.php?l=zh-tw&sd=109/09/24&ed=113/04/01&code=&choice_type=all_category&stk_cotegory=-1&disposal_measure=undefined&group_type=group_date"

request_data = get_web_content(url)
punish = pd.read_html(request_data.text)[0]
punish = punish[punish["證券代號"].isna()==False].copy()
punish = punish.iloc[:-1]
# punish["annouance_date"] = punish["公告日期"]
# punish["annouance_date"] = [str(int(i.split("/")[0]) + 1911) + i[3::] for i in punish["annouance_date"]]
# punish["end"] = [str(int(i.split("/")[0]) + 1911) + i[3::] for i in punish["end"]]
# punish["annouance_date"] = pd.to_datetime(punish["annouance_date"])
# punish["start"] = punish["start"].dt.strftime("%Y%m%d")
# punish["end"] = pd.to_datetime(punish["end"])
# punish["end"] = punish["end"].dt.strftime("%Y%m%d")

punish["trade_date"] = [str(int(i.split("/")[0]) + 1911) + i[3::] for i in punish["公告日期"]]
punish["trade_date"] = pd.to_datetime(punish["trade_date"])
punish["trade_date"] = punish["trade_date"].dt.strftime("%Y%m%d")

punish = punish.dropna(axis=1,how='all')

AttributeError: 'NoneType' object has no attribute 'text'

In [485]:
tmp = tmp.set_index(['公告日期', '證券代號']).sort_index()
punish = punish.set_index(['公告日期', '證券代號']).sort_index()

In [488]:
len(tmp.loc[punish.index])==len(punish.loc[tmp.index])

True

### parvaluechg_變更股票面額

In [31]:
path = "Y:\\TPEX\\bulletin\\parvaluechg_變更股票面額"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files if f.split('.')[0]<'20200101']

In [32]:
dfs = [pd.read_csv(os.path.join(path,f)) for f in tqdm(files)]

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 33.90it/s]


In [33]:
tmp = pd.concat(dfs, axis=0)
tmp['證券代號'] = tmp['證券代號'].astype(str)

In [34]:
def get_web_content(url):
    
    error_class = ""
    try:
        resp = requests.get(url)
    except Exception as e:
        error_class = e.__class__.__name__
        detail = e.args[0]
        print(error_class)
        print(detail)


    if error_class == "ConnectionError":
        
        print("Run Bat")
        os.startfile("VPN1.bat")
        time.sleep(15)
        
        return None
        
    
    if resp.status_code != 200:
        return None
    else:
        try:
            return resp.json()
        except:
            return None

In [35]:
cols = ['恢復買賣日期','證券代號','證券名稱','最後交易日之收盤價格','恢復買賣開始日參考價','漲停價格','跌停價格','開始交易基準價','詳細資料']

In [36]:
# 測20200101後 嚴謹測
url = "https://www.tpex.org.tw/web/bulletin/parvaluechg/rslt_result.php?l=zh-tw&d=108/09/01&ed=112/09/07"

request_data = get_web_content(url)

try:
    writing_data = pd.DataFrame()
    writing_data = pd.DataFrame(request_data['aaData'])
except:
    writing_data = pd.DataFrame()

if writing_data.shape[0] > 0:
           
    writing_data.columns = cols

In [37]:
tmp = tmp.set_index(['恢復買賣日期', '證券代號']).sort_index()
writing_data = writing_data.set_index(['恢復買賣日期', '證券代號']).sort_index()

In [38]:
len(tmp.loc[writing_data.index])==len(writing_data.loc[tmp.index])

True

### dailyquo_除權除息計算結果表

In [40]:
path = "Y:\\TPEX\\exright\\dailyquo_除權除息計算結果表"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files if f.split('.')[0]<'20200101']

In [41]:
dfs = [pd.read_csv(os.path.join(path,f)) for f in tqdm(files) if f.split('.')[0]>='20180101']

100%|█████████████████████████████████████████████████████████████████████████████| 1720/1720 [00:12<00:00, 133.21it/s]


In [42]:
tmp = pd.concat(dfs, axis=0)
tmp['股票代號'] = tmp['股票代號'].astype(str)

In [43]:
cols_1 = ["除權息日期", "股票代號", "股票名稱", "除權息前收盤價", "除權息參考價", "權值", "息值", "權值+息值", "權/息", 
        "漲停價", "跌停價", "開始交易基準價", "減除股利參考價", "現金股利", "每仟股無償配股", "現金增資股數", "現金增資認購價", 
        "公開承銷股數", "員工認購股數", "原股東認購股數", "按持股比例仟股認購"]

cols_2 = ["除權息日期", "股票代號", "股票名稱", "除權息前收盤價", "除權息參考價", "權值", "息值", "權值+息值", "權/息", 
        "漲停價", "跌停價", "開始交易基準價", "減除股利參考價", "現金股利", "每仟股無償配股", "員工紅利轉增資","現金增資股數", "現金增資認購價", 
        "公開承銷股數", "員工認購股數", "原股東認購股數", "按持股比例仟股認購"]
# get_dt_YYYYMMDD >= "201601011
cols = cols_1

In [44]:
# 測20200101後 嚴謹測
url = "https://www.tpex.org.tw/web/stock/exright/dailyquo/exDailyQ_result.php?l=zh-tw&d=107/01/01&ed=113/03/01"

request_data = get_web_content(url)

try:
    writing_data = pd.DataFrame()
    writing_data = pd.DataFrame(request_data['aaData'])
except:
    writing_data = pd.DataFrame()

if writing_data.shape[0] > 0:
           
    writing_data.columns = cols

In [45]:
writing_data2 = writing_data.copy()

In [46]:
tmp = tmp.set_index(["除權息日期", "股票代號"]).sort_index()
writing_data2 = writing_data2.set_index(["除權息日期", "股票代號"]).sort_index()

In [47]:
for i in writing_data2[~writing_data2.index.isin(tmp.index)].index:
    date_ = i[0]
    file = str(int(date_.split('/')[0])+1911)+date_.split('/')[1]+date_.split('/')[2]+'.csv'
    print(i)
    # writing_data[writing_data['除權息日期']==date_].reset_index(drop=True).to_csv(os.path.join(path, file))

('109/11/20', '006201')
('110/11/19', '006201')
('112/10/04', '020035')


### revivt_減資恢復交易參考價

In [48]:
path = "Y:\\TPEX\\exright\\revivt_減資恢復交易參考價"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files if f.split('.')[0]<'20200101']

In [49]:
dfs = [pd.read_csv(os.path.join(path,f)) for f in tqdm(files) if f.split('.')[0]>='20180101']

100%|███████████████████████████████████████████████████████████████████████████████| 207/207 [00:01<00:00, 172.40it/s]


In [50]:
tmp = pd.concat(dfs, axis=0)
tmp['股票代號'] = tmp['股票代號'].astype(str)

In [51]:
cols = ["恢復買賣日期", "股票代號", "股票名稱", "最後交易日之收盤價格", "減資恢復買賣開始日參考價格", 
        "漲停價格", "跌停價格", "開始交易基準價", "除權參考價", "減資原因", "詳細資料"]

In [52]:
# 測20200101後 嚴謹測
url = "https://www.tpex.org.tw/web/stock/exright/revivt/revivt_result.php?l=zh-tw&d=107/01/01&ed=112/09/08"

request_data = get_web_content(url)

try:
    writing_data = pd.DataFrame()
    writing_data = pd.DataFrame(request_data['aaData'])
except:
    writing_data = pd.DataFrame()

if writing_data.shape[0] > 0:
           
    writing_data.columns = cols

In [53]:
writing_data2 = writing_data.copy()

In [54]:
tmp = tmp.set_index(["恢復買賣日期", "股票代號"]).sort_index()
writing_data2 = writing_data2.set_index(["恢復買賣日期", "股票代號"]).sort_index()

In [55]:
for i in writing_data2[~writing_data2.index.isin(tmp.index)].index:
    date_ = i[0]
    file = str(int(str(date_)[:-4])+1911)+str(date_)[-4:]+'.csv'
    print(i)
    # writing_data[writing_data['恢復買賣日期']==date_].reset_index(drop=True).to_csv(os.path.join(path, file))

### minute_index

In [56]:
path = "Y:\\TPEX\\iNdex_info\\minute_index"
files = sorted(os.listdir(path))
# [f for f in check_files if f not in files]

In [57]:
tw = pd.read_csv(r'Y:\yahoo_data\^TWOII.csv')

In [58]:
# 測20200101後 嚴謹測
for f in tqdm(files):
    date_ = f.split('.')[0]
    if date_<='20200101':
        continue
    with open(os.path.join(path,f), "r", encoding="utf-8") as fa:
        lines = fa.readlines()
        lines = [l.replace('"','').replace(' ','') for l in lines]
        lines = [l.split(',')[1:25] for l in lines][2:-1]
    tmp = pd.DataFrame(lines[1:], columns=lines[0])

    if tw[tw['Date'].apply(lambda x: x.replace('-',''))==date_].shape[0]==0:
        continue
    if abs(float(tmp['櫃買指數'].iloc[-1].replace(',',''))/tw[tw['Date'].apply(lambda x: x.replace('-',''))==date_]['Close'].iloc[0]-1)>0.0001:
        print(date_)

100%|██████████████████████████████████████████████████████████████████████████████| 1505/1505 [01:16<00:00, 19.66it/s]


In [174]:
tmp['櫃買指數'].iloc[-1]

'217.34'

In [175]:
tw

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2005-01-03,6166.367591,6183.127295,6129.257375,6143.097656,4212000,0,0
1,2005-01-04,6117.847443,6129.357166,6053.157742,6060.437500,3213200,0,0
2,2005-01-05,5999.447712,6030.167324,5988.347656,5988.347656,2850000,0,0
3,2005-01-06,5988.698218,6002.497972,5971.558144,5982.098145,2381000,0,0
4,2005-01-07,5990.757610,6007.317607,5934.808110,5935.968262,2991000,0,0
...,...,...,...,...,...,...,...,...
4640,2023-12-08,17309.359375,17465.349609,17309.359375,17383.990234,3155300,0,0
4641,2023-12-11,17416.960938,17451.820312,17372.820312,17418.339844,3003300,0,0
4642,2023-12-12,17429.609375,17528.720703,17404.880859,17450.630859,3356500,0,0
4643,2023-12-13,17447.509766,17507.919922,17438.689453,17468.929688,3410000,0,0
